---
title: Supporting material
numbering:
  headings: false
---

Companion to the paper. Each appendix opens with its job and states what the main text keeps, what the appendix carries and what the repository keeps.


In [1]:
import paper_paths  # noqa: F401  (puts the repository on sys.path)

(appendix-a)=
## Appendix A. Notation and terms

**Job.** To say exactly what each symbol or term means in this paper, in one table compiled from the glossary.

- **Main text keeps:** {numref}`section %s <sect-notation>`'s notation paragraph and the symbols introduced where first used.
- **This appendix carries:** one table: term, symbol, meaning in this paper, what it is not, and where it is defined, compiled from `glossary.yaml` and never typed.
- **The repository keeps:** the register's reasoning and its phrase lists (`docs/superpowers/notes/notation.md`).

In [2]:
from generate_register import register_table

register_table()

Term,Symbol,Meaning in this paper,Not,Defined in
configuration,$c$,the design variables of one evaluated design,an input of the fit; the fit sees attributes and metrics only,Prediction from configuration
attributes,$a$,"the evaluation outcomes of a configuration; the evaluation is a black box (simulator, physical test or human assessment)","metrics; attributes are not conditioned, positive or oriented",A4
metrics,$m \in \mathbb{R}^d_{++}$,"the conditioned attributes, a point of the positive orthant",the raw attributes; zero is tolerated in data and masked in the fit,"A1, A4, L9"
log-metrics,$u = \log m$,the componentwise logarithm of the metrics,the chart coordinates Z,"L1, A1"
conditioning,,"the log-affine, invertible map from attributes to metrics",part of the fit; the fit consumes metrics,"A4, L9"
number of designs,$N$,the number of evaluated configurations,a sample drawn from the attainable set,The model
number of metrics,$d$,the dimension of the trade space,the number of configuration variables,"The model, A0"
number of terms,$K$,the number of monomials of the posynomial frontier,a selected or estimated quantity,"L10, A6"
invariant value,$V(m) = \prod_j m_j^{w_j}$,the weighted geometric mean of the metrics,a probability or a bounded score; V above 1 is possible at finite temperature,"L1, A1, A2"
exponent vector,$w_k$,a point on the simplex over the d metrics,a loading vector with sign or unit norm,"A2, L3, L4"


(appendix-b)=
## Appendix B. Proofs, step for step with their scripts

**Job.** To let a reader follow each proof in ordinary mathematics and, at any step, open the exact line of code that checks it.

- **Main text keeps:** each statement as a theorem-style block with hypotheses, a narrative proof sketch and a one-line pointer to its file.
- **This appendix carries:** per proof, in the paper's numbering: the statement with its hypotheses, its file, the proof as numbered steps (a sentence of prose, the step's formal claim, its kind and a link to its line), and the ledger status; compiled from `derivation/proof_map_record.json`, the lemma scripts and the axiom ledger, with the checker's table at the end.
- **The repository keeps:** the scripts, the axiom ledger, the runner, the checker and the committed record.

**What the scripts establish.** The lemma scripts are a substitute for pencil-and-paper derivation, not formal verification. They sit between a free-form proof, which no tool checks, and a proof assistant, whose language and friction put the derivation beyond most readers: each step is one line of symbolic algebra that the reader can run against its sentence, on the stack the paper already uses, so rigor is bought at that friction. A step checks the algebra it states under the ledger's assumptions; it does not check that those assumptions hold for a table of designs ({ref}`Appendix E <appendix-e>`). The formal prover in this work is the Z3 solver behind the requirement checks of the case's system model (the repository tier of {ref}`Appendix C <appendix-c>`), and formal verification is said only of those checks.

In [3]:
from generate_proofs import how_to_read

how_to_read()

"Each proof is one file of `derivation/`, and each step of the proof is one line of that file. The address of a step is its file and its tag (`L2.1`, `L2.2`, ...), and the tag is unique within the file. The record holds 22 results in 20 files and 505 steps. Every step is a line of code, including a step that cites a result, and is of one of three kinds:\n\n- **computed** (333 steps): SymPy derives the step's condition and the line asserts it.\n- **by a declared axiom** (160 steps): the line applies an entry of the axiom ledger that the script imported by name; the entry's wording and citation are shown beside the step, and the link gives the import line, the declaration's address in the proof's own file.\n- **by a proved result** (12 steps): the line applies the result of an earlier proof that the script imported; the step names that proof's statement and links to its entry.\n\nThere is no kind for an argument made only in prose. A step that SymPy cannot derive is an explicit hypothesis or an entry of the ledger. Two results may share a file (L14 proves Proposition 3 and its corollary, the approximate certificate): each names the same file and owns its own steps. The appendix has two parts:\n\n- The fit and its geometry (L1 to L11)\n- The solver for K > 1 and its three levels of guarantee (L12 to L20)\n\nWhat the correspondence establishes: every step of the prose has exactly one line of code, the lines are in the order of the prose, every computed line passes, and neither a line nor a sentence can change without the checker reporting the step. What it cannot establish is that a sentence means what its line computes; that reading is the human validation, and the tool guarantees there is always exactly one line to read it against."

In [4]:
from generate_proofs import proof_map

proof_map()

Result,File,Steps,Computed,By axiom,By proved result
"L1. Homogeneity and invariant form: V(k m) = k V(m), V(m / V) = 1",lemma_01_homogeneity.py,8,4,4,0
L2. Uniform shift: J_tau(w; U + c 1^T) = J_tau(w; U) + c,lemma_02_shift.py,8,4,4,0
L3. gradient of the soft-maximum objective and the full simplex KKT conditions,lemma_03_gradient_kkt.py,35,26,9,0
L4. convexity of the soft-maximum objective (second directional derivative = tau Var_p),lemma_04_convexity.py,24,14,9,1
L5. LSE = LME + log(n)/tau; mean <= LME <= max; tau limits,lemma_05_lse_lme.py,25,17,8,0
L6. Gaussian cumulant of a weighted noise sum,lemma_06_gaussian_cumulant.py,9,6,3,0
L7. Elasticities -w_k/w_j on the frontier; improvement factor V^(-1/w_j),lemma_07_elasticities.py,9,6,2,1
L8. Sum-zero (Helmert-type) chart and scale/shape bijection,lemma_08_helmert_chart.py,14,9,5,0
L9. Pull-back: V = e^{w.b - beta} prod_j (a_j + s_j)^{w_j alpha_j},lemma_09_pullback.py,7,3,4,0
"L10. Extension Phi_K: shift equivariance, K = 1 reduction, kappa -> infinity = max",lemma_10_extension.py,25,15,10,0


(sub-b-1)=
### B.1 The fit and its geometry (L1 to L11)

In [5]:
from generate_proofs import proofs_fit

proofs_fit()

Step,Kind,Prose of the step,Line checked (the script's text) and link
L1.1,by declared axiom A2 (posited): the exponents w lie on the simplex: w_j >= 0 and sum_j w_j = 1; declared at line 22,A2: the simplex sum_j w_j is replaced by 1 wherever it appears,return A2.rewrite(expr)line 37
L1.2,"by declared axiom A1 (posited): metrics are strictly positive, larger-is-better: m_i[j] > 0, so u = log m is defined; declared at line 22",A1 (m_j > 0) licenses log(m_j^w_j) = w_j log m_j: V is exp of a Sum,"V_prod_as_exp = product_to_exp_sum(V_prod, A1)line 50"
L1.3,"by declared axiom A1 (posited): metrics are strictly positive, larger-is-better: m_i[j] > 0, so u = log m is defined; declared at line 22",A1 again for the scaled point k m: (k m_j)^w_j in log space,"Vk_exp = product_to_exp_sum(sp.Product((k * m[j]) ** w[j], lim), A1)line 54"
L1.4,computed,V(m) = prod_j m_j^w_j equals exp(sum_j w_j log m_j),"assert sp.simplify(r['V_prod_as_exp'] - V) == 0, 'L1 FAIL: V is not exp(sum w log m)'line 83"
L1.5,computed,"homogeneity of degree 1: V(k m) = k V(m), using sum_j w_j = 1","assert sp.simplify(r['Vk'] - S.k_scale * V) == 0, 'L1 FAIL: V(k m) != k V(m)'line 85"
L1.6,computed,"invariant form: V(m / V(m)) = 1, the point lies on the frontier","assert sp.simplify(r['V_inv_form'] - 1) == 0, 'L1 FAIL: V(m / V(m)) != 1'line 87"
L1.7,computed,"offset: V(m e^-beta) = e^-beta V(m), so log V = w.u - beta","assert sp.simplify(r['V_tilde'] - sp.exp(-r['beta0']) * V) == 0, 'L1 FAIL: V(m e^-beta) != e^-beta V(m)'line 89"
L1.8,"by declared axiom A1 (posited): metrics are strictly positive, larger-is-better: m_i[j] > 0, so u = log m is defined; declared at line 22","the degree of homogeneity passes through sum_j w_j (without it, not degree 1)","assert S.simplex.lhs in linear(product_to_exp_sum(sp.Product((S.k_scale * S.m[S.j]) ** S.w[S.j], (S.j, 1, S.d)), A1)).atoms(sp.Sum), 'L1 FAIL: homogeneity must pass through sum_j w_j'line 94"
Step,Kind,Prose of the step,Line checked (the script's text) and link
L2.1,by declared axiom A2 (posited): the exponents w lie on the simplex: w_j >= 0 and sum_j w_j = 1; declared at line 19,A2: sum_j w_j = 1 turns tau sum_j w_j (u_ij + c) into tau s_i + tau c,"term_lin = A2.rewrite(linear(tau * sp.Sum(w[j] * (u[i, j] + c), (j, 1, d))))line 43"


(sub-b-2)=
### B.2 The several-term fit and its levels of guarantee (L12 to L20)

**The algorithm.** The model is $\Phi_K$ (L10); every term is fitted against every design, each offset $\beta_k=J_\tau(Uw_k)$ being the support over all rows; the fit minimizes the mean slack. The alternation takes weights from the current model and then solves one program for all terms (L16). The package uses the sharp limit of the weights, one label per design, $k(i)=\arg\max_ks_{ik}$ with $s_{ik}=w_k\cdot u_i-\beta_k$; the program then separates into one convex program per term, $f_k(w)=J_\tau(Uw)-\bar u_k\cdot w$, with $n_k$ the designs labelled $k$ and $\bar u_k$ their mean log-metric. For each restart the package starts from random centres in the chart and repeats: (1) the labels of the current model; (2) the refit $w_k=\arg\min_{w\in\Delta}f_k(w)$ over all rows, $\beta_k=J_\tau(Uw_k)$; (3) if the mean slack rose, stop with the previous state; until the labels are fixed or a limit is reached; it returns the restart with the smallest mean slack. L16 states the program for weights $\alpha_{ik}=\mathrm{softmax}_k(\kappa s_{ik})$: the fixed-weight objective bounds the soft model's mean slack from above and equals it at the model the weights came from, and it is a generalized posynomial in $e^{w}$, so its minimization is a generalized geometric program in the sense of Boyd et al. [@boyd2007tutorial]. L16 does not prove that the alternation converges. L17 treats the soft fit at finite $\tau$ and $\kappa$: the gradient and the Hessian $D-\kappa M$ of its objective, and the second-order test that certifies a strict local minimizer. The label belongs to the package and not to the model; which term scores a design highest is analysis of the fitted model.

In [6]:
from generate_proofs import proofs_solver

proofs_solver()

Step,Kind,Prose of the step,Line checked (the script's text) and link
L12.1,by declared axiom A2 (posited): the exponents w lie on the simplex: w_j >= 0 and sum_j w_j = 1; declared at line 40,A2: the weight families v and w on the simplex are nonnegative,"vn, wn = (A2.nonneg_base('vn'), A2.nonneg_base('wn'))line 78"
L12.2,"by declared axiom F12 (posited): min representation for L12 and L13: the minimum m of a finite family x_j (the gradient components g_j in L12, the row gaps in L13) is a symbol with x_j >= m for all j and x_{j0} = m for some j0; declared at line 40",F12: the slacks gm_j = g_j - m are nonnegative,gm = F12.nonneg_base('gm')line 80
L12.3,by declared axiom A2 (posited): the exponents w lie on the simplex: w_j >= 0 and sum_j w_j = 1; declared at line 40,A2: sum_j v_j = 1 gives g.v - m = sum_j v_j (g_j - m),"gv_minus_m = A2.rewrite(linear(sp.Sum(vn[j] * (gm[j] + m), lim) - m), rename={w: vn})line 83"
L12.4,by declared axiom A2 (posited): the exponents w lie on the simplex: w_j >= 0 and sum_j w_j = 1; declared at line 40,A2: sum_j w_j = 1 gives lambda - m = w.g - m = sum_j w_j (g_j - m) = G,"lam_minus_m = A2.rewrite(linear(sp.Sum(wn[j] * (gm[j] + m), lim) - m), rename={w: wn})line 88"
L12.5,"by declared axiom I8 (inherited): first-order convexity inequality: a differentiable convex f satisfies f(v) >= f(w) + grad f(w).(v - w) for all feasible v, w [Boyd & Vandenberghe, Convex Optimization, section 3.1.3]; declared at line 40",I8 at v = v*: the slack a = f* - f(w) - g.(v* - w) is >= 0,a_nn = I8.slack_symbol('a_nn')line 104
L12.6,"by declared axiom I8 (inherited): first-order convexity inequality: a differentiable convex f satisfies f(v) >= f(w) + grad f(w).(v - w) for all feasible v, w [Boyd & Vandenberghe, Convex Optimization, section 3.1.3]; declared at line 40",I8: the slack equation gives f* = f(w) + g.(v* - w) + a,"f_star = sp.solve(sp.Eq(I8.slack_expr(f_v=fs, f_w=fw, g_dv=gvs - gw), a_nn), fs)[0]line 106"
L12.7,"by declared axiom F12 (posited): min representation for L12 and L13: the minimum m of a finite family x_j (the gradient components g_j in L12, the row gaps in L13) is a symbol with x_j >= m for all j and x_{j0} = m for some j0; declared at line 40",F12: step 2 at v = v*: the slack b = g.v* - m is >= 0,b_nn = F12.nonneg_symbol('b_nn')line 108
L12.8,"by declared axiom I12 (inherited): complementary slackness of the LP pair at tau = infinity: the dual weights p are supported on the rows whose score attains the maximum M, p_i (M - s_i) = 0 for every row i, with s_i = w.u_i [Boyd & Vandenberghe, Convex Optimization, section 5.5.2 (complementary slackness)]; declared at line 40",I12: complementary slackness p_i (M - s_i) = 0 applied to each summand: the sum is 0,support_zero = I12.rewrite(support).doit()line 127
L12.9,computed,exact two-metric LP: primal and dual values and gaps at an optimal pair and off it,"lp = dict(primal_at_optimum=f_lp(w_opt), dual_at_optimum=dual(p_opt), gap_at_optimum=f_lp(w_opt) - dual(p_opt), f_star=f_lp(w_opt), f_off_optimum=f_lp(w_off), gap_off_optimum=f_lp(w_off) - dual(p_off), active_off=Uc * w_off)line 150"
L12.10,by proved result L4: convexity of the soft-maximum objective (second directional derivative = tau Var_p) (see L4; imported at line 41),"by L4: J_tau is convex along every line, so f = J_tau - linear is convex","convex = L4_RESULT.holds(ctx, 'convex') and second_linear == 0line 160"


(sub-b-3)=
### B.3 The checker and the record

In [7]:
from generate_proofs import checker

checker()

"The checker passes only when: every proof names one file; every step of every file belongs to exactly one proof; the prose steps and the script steps correspond one to one and in order; every step by a declared axiom applies an entry that exists in the ledger and that the file imports; the entries a file imports, applies and lists as used are the same set; a proved result is imported from an earlier proof and applied by a step; no premise hides in a signed symbol; and the fingerprint of each step (its tag, prose and line of code) and of each ledger entry equals the record's.\n\n| Item | Value |\n|---|---|\n| Record | `derivation/proof_map_record.json` |\n| Record version | 2 |\n| SHA-256 of the record | `1d7b83c32a20cd1d68284b5eb042b15095cdff8395fb3cb410deee69300933f5` |\n| Lemma files, steps | 20, 505 |\n| Ledger entries fingerprinted | 37 |\n| Check the scripts against the record | `uv run python -m derivation.proofmap` |\n| Also run every computed step | `uv run python -m derivation.proofmap --execute` |\n| Refresh the record (deliberate; prints what moved) | `uv run python -m derivation.proofmap --update` |\n\n| File | Ledger entries imported | Proved results imported | Exported | Steps | Signed symbols declared as hypotheses |\n|---|---|---|---|---|---|\n| `lemma_01_homogeneity.py` | A1, A2 | none | L1_RESULT | 8 | 0 |\n| `lemma_02_shift.py` | A1, A2, A3 | none | none | 8 | 0 |\n| `lemma_03_gradient_kkt.py` | A2, I10, R1 | none | L3_RESULT | 35 | 0 |\n| `lemma_04_convexity.py` | A1, A2, A3, I11, I13 | L3_RESULT | L4_RESULT | 24 | 0 |\n| `lemma_05_lse_lme.py` | A3, F5 | none | none | 25 | 0 |\n| `lemma_06_gaussian_cumulant.py` | A2, A3, A5, I5 | none | none | 9 | 0 |\n| `lemma_07_elasticities.py` | A1, A2 | L1_RESULT | none | 9 | 0 |\n| `lemma_08_helmert_chart.py` | A7, I6 | none | none | 14 | 0 |\n| `lemma_09_pullback.py` | A2, A4 | none | none | 7 | 0 |\n| `lemma_10_extension.py` | A1, A6, F10 | none | none | 25 | 0 |\n| `lemma_11_pareto.py` | A2 | none | none | 20 | 5 |\n| `lemma_12_term_gap.py` | A2, F12, I1, I12, I8 | L4_RESULT | L12_RESULT | 33 | 1 |\n| `lemma_13_margin.py` | F12, I9, R2 | none | L13_RESULT | 17 | 10 |\n| `lemma_14_assignment_stable.py` | A8, R2 | L12_RESULT, L13_RESULT | L14_RESULT | 20 | 1 |\n| `lemma_15_descent.py` | A8, F12 | L12_RESULT, L14_RESULT | none | 22 | 0 |\n| `lemma_16_soft_fit_alternation.py` | A1, A2, A3, A6, I14, I15 | none | none | 49 | 0 |\n| `lemma_17_soft_fit_certificate.py` | A1, A2, A3, A6, I16 | none | none | 52 | 0 |\n| `lemma_18_noise_covariance.py` | A10, A3, A4, A5, A9 | none | L18_RESULT | 21 | 0 |\n| `lemma_19_noise_shift_bound.py` | A11, A3, F19, F5, I17 | none | none | 44 | 2 |\n| `lemma_20_noise_ridge.py` | A1, A3, A5, A9, I18, I9, R3 | L18_RESULT, L3_RESULT, L4_RESULT | L20_RESULT | 63 | 4 |"

(appendix-c)=
## Appendix C. The case's model: how the table of metrics came to be

**Job.** To let a reader judge whether the case, a cubesat model coupling power, mass and attitude control, is a fair test, and build a comparable one, by following the model from configurations to the table of metrics; what the method found in that table is {ref}`Appendix D <appendix-d>`. The cubesat case is a worked example of design optimization, not a study of either use, and in this design loop the simulation is how the attributes are measured, while the measured attributes of a fielded system would enter the same table and the same fit, because the evaluation is a black box to the method ({numref}`Section %s <sect-notation>`).

- **Main text keeps:** {numref}`section %s <sect-case-construction>` in about 300 words: the cubesat model, the two regimes, the eight attributes, feasibility and conditioning.
- **This appendix carries:** tables generated from the model and never typed: the configuration space, the evaluation with the part of the model that produces each attribute, feasibility, conditioning, the table of metrics, the datasets with their provenance, and the planted example.
- **The repository keeps:** the SysML model file, the evaluation code, the experiment ledger and the data.

(sub-c-1)=
### C.1 The configuration space

In [8]:
from generate_model import configuration_space

configuration_space()

'| Design variable | Declared min | Declared max | cubesat-primary min | cubesat-primary max | cubesat-stock min | cubesat-stock max |\n|---|---|---|---|---|---|---|\n| panel_span_m | 0.3 | 1 | 0.3006 | 0.9992 | 0.3006 | 0.9992 |\n| battery_wh | 20 | 200 | 20.02 | 199.8 | 20.02 | 199.8 |\n| Kp | 0.2 | 2 | 0.2005 | 1.999 | 0.03011 | 0.999 |\n| Kd | 0.3 | 1.5 | 0.3001 | 1.5 | 0.3001 | 1.5 |\n\n| Covariate | Ordered levels | cubesat-primary designs per level | cubesat-stock designs per level |\n|---|---|---|---|\n| wheel_class | small, medium, large | small: 344, medium: 345, large: 311 | small: 344, medium: 345, large: 311 |\n| st_grade | coarse, medium, fine | coarse: 347, medium: 342, fine: 311 | coarse: 347, medium: 342, fine: 311 |'

(sub-c-2)=
### C.2 The evaluation

In [9]:
from generate_model import evaluation

evaluation()

'| Attribute | Column | Unit | Larger is better |\n|---|---|---|---|\n| settling time | settling_time_s | s | no |\n| pointing error | pointing_error_deg | deg | no |\n| total mass | total_mass_kg | kg | no |\n| control effort | control_effort | torque.s, proxy | no |\n| power margin | power_margin_w | W | yes |\n| energy balance | energy_balance_wh | Wh | yes |\n| peak slew power | peak_slew_power_w | W | no |\n| depth of discharge | dod_eclipse | fraction | no |\n\n| Scenario constant (column) | Value |\n|---|---|\n| slew_deg | 30 |\n| t_end_s | 600 |\n| max_step_s | 0.5 |'

In [10]:
from generate_model import attribute_sources

attribute_sources()

Attribute,File,Name in the file,How it is obtained
settling time,adcs_sim.py,simulate_adcs,measured: on the integrated slew
pointing error,cubesat_model.py,pointing_error_deg,computed: pointing budget
total mass,cubesat_tradespace.sysml,MassAccounting,computed: accounting
control effort,adcs_sim.py,simulate_adcs,measured: on the integrated slew
power margin,cubesat_tradespace.sysml,PowerMargin,computed: accounting
energy balance,cubesat_tradespace.sysml,EnergyBalance,computed: accounting
peak slew power,cubesat_tradespace.sysml,PeakSlewPower,computed: accounting
depth of discharge,cubesat_tradespace.sysml,DepthOfDischarge,computed: accounting


(sub-c-3)=
### C.3 Feasibility

In [11]:
from generate_model import feasibility

feasibility()

condition,fails alone,removed in this order,remaining
settled,24,24,976
power_ok,0,0,976
momentum_ok,188,165,811
dod_ok,131,105,706
all attributes positive,0,0,706
condition,fails alone,removed in this order,remaining
settled,22,22,978
power_ok,0,0,978
momentum_ok,81,81,897
dod_ok,131,117,780


(sub-c-4)=
### C.4 Conditioning

In [12]:
from generate_model import conditioning

conditioning()

attribute,orientation,raw min,raw max,log-spread k,std of log m,m min,m max
settling time,reciprocal,7.324,506.1,0.8271,1,0.05523,9.255
pointing error,reciprocal,0.0006016,0.008639,0.8604,1,0.3139,6.946
total mass,reciprocal,11.9,16.06,0.06581,1,0.07706,7.388
control effort,reciprocal,0.008614,0.3449,0.745,1,0.07332,10.38
power margin,direct,6.866,65.85,0.5133,1,0.08544,6.992
energy balance,direct,1.186,55.9,0.8275,1,0.04783,5.033
peak slew power,reciprocal,1.304,7.4,0.6565,1,0.2554,3.591
depth of discharge,reciprocal,0.03995,0.2997,0.5726,1,0.1823,6.155
attribute,orientation,raw min,raw max,log-spread k,std of log m,m min,m max
settling time,reciprocal,7.555,598.2,0.9207,1,0.1296,14.95


(sub-c-5)=
### C.5 The table of metrics

In [13]:
from generate_model import metric_table

metric_table()

design,settling time,pointing error,total mass,control effort,power margin,energy balance,peak slew power,depth of discharge
0,1.091,1.859,0.2572,1.656,0.9351,1.299,0.2554,1.342
1,0.1276,1.118,1.176,0.554,1.596,1.529,2.191,4.178
2,0.3706,1.148,5.801,1.836,0.2073,0.08689,2.191,0.3036
3,0.5873,0.3232,5.912,4.216,0.9285,0.9517,3.591,0.1965
4,2.429,0.3201,0.5247,1.076,0.6854,1.078,0.3263,2.29
design,settling time,pointing error,total mass,control effort,power margin,energy balance,peak slew power,depth of discharge
0,0.3212,0.41,0.2446,3.123,1.019,1.139,0.5293,1.338
1,2.849,2.41,0.947,0.3013,3.35,2.857,1.684,1.167
2,1.095,1.411,1.179,0.8689,1.326,1.337,1.998,4.218
3,0.5597,1.075,1.205,0.9984,4.851,3.748,1.998,1.026


The association of every pair of traded metrics over the feasible designs (`case.association_all_pairs`) is the step that would find two metrics the data cannot tell apart: a pair at |Spearman| 0.99 or above is named one axis, and its two exponents are read summed per term in {ref}`D.3 <sub-d-3>` and {ref}`D.4 <sub-d-4>`.

In [14]:
from generate_model import metric_associations

metric_associations()

metric a,metric b,"Spearman, cubesat-primary","Pearson, cubesat-primary","Spearman, cubesat-stock","Pearson, cubesat-stock",one axis
power margin,energy balance,+0.982,+0.960,+0.988,+0.966,no
total mass,peak slew power,+0.865,+0.841,+0.818,+0.777,no
settling time,peak slew power,-0.628,-0.631,-0.235,-0.306,no
settling time,total mass,-0.588,-0.574,-0.160,-0.172,no
pointing error,peak slew power,-0.484,-0.284,-0.360,-0.304,no
total mass,energy balance,-0.482,-0.456,-0.445,-0.420,no
total mass,control effort,+0.393,+0.398,+0.286,+0.290,no
settling time,control effort,-0.383,-0.300,-0.837,-0.838,no
total mass,power margin,-0.369,-0.362,-0.356,-0.348,no
control effort,energy balance,-0.342,-0.325,-0.288,-0.282,no


(sub-c-6)=
### C.6 The datasets

In [15]:
from generate_model import datasets

datasets()

Label,Regime,File,Ledger entry,Command,Designs sampled,Designs kept,SHA-256,Ledger hash
cubesat-primary,primary,examples/adcs/data/v2/adcs_designs_cubesat_gains.csv,v2-primary@8,PYTHONPATH=examples/adcs uv run python -m experiments run v2-primary --supersede,1000,706,a43ccc557bc6a572be315309fbe5f5b101df504fb38c2a2692f12ba15c4c1e2f,a43ccc557bc6a572be315309fbe5f5b101df504fb38c2a2692f12ba15c4c1e2f
cubesat-stock,stock,examples/adcs/data/v2/adcs_designs_cubesat.csv,v2-baseline@8,PYTHONPATH=examples/adcs uv run python -m experiments run v2-baseline --supersede,1000,780,10ae79a224b5a0b370927d2419ca238e634d25f434a044385f920a3c1a77ee74,10ae79a224b5a0b370927d2419ca238e634d25f434a044385f920a3c1a77ee74
planted-3-term,synthetic,none,none,"make_frontier_data(n_samples=900, n_features=4, n_terms=3, exponents=[[0.5, 0.5, 0, 0], [0, 0.5, 0.5, 0], [0, 0, 0.5, 0.5]], noise=0.0, random_state=0)",900,900,b17d4f8fdd4679239900073994631fb76e806d84599f8805f1c4089d9e8e7add,none


(sub-c-7)=
### C.7 The planted example

In [16]:
from generate_model import planted_example

planted_example()

'Dataset planted-3-term:\n\n| Argument of make_frontier_data | Value |\n|---|---|\n| n_samples | 900 |\n| n_features | 4 |\n| n_terms | 3 |\n| noise | 0 |\n| random_state | 0 |\n\n| Planted term | metric 1 | metric 2 | metric 3 | metric 4 |\n|---|---|---|---|---|\n| term 1 | 0.5 | 0.5 | 0 | 0 |\n| term 2 | 0 | 0.5 | 0.5 | 0 |\n| term 3 | 0 | 0 | 0.5 | 0.5 |'

(appendix-d)=
## Appendix D. Extended results: what the method found in the table

**Job.** To let a reader check that the main text's summary of the analysis is fair, including the results that are mixed; how the table of metrics came to be is {ref}`Appendix C <appendix-c>`.

- **Main text keeps:** {numref}`section %s <sect-case-analysis>` in about 500 words: K = 1, K = 3, the regime comparison, the candidates, propose-predict-check in one paragraph and the nondominated counts.
- **This appendix carries:** tables that cite their dataset by the label {ref}`Appendix C <appendix-c>` defines: the temperature selection, the diagnostics at K = 1, 2, 3, the exponents and terms, the regime comparison, the sensitivity variants, the refits by category, the candidate designs (the table of the candidates, which the main text leaves here), the solver's termination records with what each status asserts, the nondominated counts, propose-predict-check, the stability of the conclusions under rounding-level jitter in the design variables, the conditioning of the exponents, the refit on a term's own rows, the recovery of planted terms over twenty tables, the mean slack at a fixed temperature and a principal component analysis of the near-frontier chart, and the four threshold requirements read after the fit with the fit on three design sets.
- **The repository keeps:** the case module, the record scripts and the full printed output.

(sub-d-1)=
### D.1 Selection of the temperature

In [17]:
from generate_results import temperature_selection

temperature_selection()

tau,frac_above,n_eff,metrics with w>1e-3,largest w,largest at
3,0.4476,561.3,6,0.3119,total mass
5.087,0.3711,412.7,6,0.3183,total mass
8.626,0.2989,229.8,6,0.3282,total mass
14.63,0.2153,100.8,6,0.343,total mass
24.8,0.1416,43.67,5,0.3538,total mass
42.06,0.08499,20.06,5,0.3582,total mass
71.32,0.04958,9.995,5,0.3587,total mass
120.9,0.02691,5.668,5,0.3576,total mass
205.1,0.01983,3.903,5,0.3559,total mass
347.8,0.01416,3.342,5,0.3544,total mass


(sub-d-2)=
### D.2 Diagnostics at K = 1, 2, 3

In [18]:
from generate_results import k_diagnostics

k_diagnostics()

K,tau,band 1/tau,objective_,frac_above_,term rows,term shares,max gap,status,gamma,radius,bound
1,38.1,0.0262,0.2434,0.09207,[706],[1.0],8.696e-06,not_applicable,nan,nan,8.696e-06
2,94.42,0.0106,0.1823,0.0949,"[414, 292]","[0.8, 0.2]",below 1e-12,certified,0.000697,6.95e-05,below 1e-12
3,113.2,0.0088,0.1458,0.09915,"[181, 162, 363]","[0.034, 0.241, 0.724]",below 1e-12,certified,0.00121,0.0001209,below 1e-12
K,tau,band 1/tau,objective_,frac_above_,term rows,term shares,max gap,status,gamma,radius,bound
1,45.68,0.0219,0.1786,0.1038,[780],[1.0],1.24e-05,not_applicable,nan,nan,1.24e-05
2,71.92,0.0139,0.1417,0.1,"[454, 326]","[0.562, 0.438]",below 1e-12,certified,0.001413,0.0001327,below 1e-12
3,113.2,0.0088,0.09269,0.09103,"[253, 185, 342]","[0.115, 0.462, 0.423]",below 1e-12,certified,0.00035,3.29e-05,below 1e-12


(sub-d-3)=
### D.3 Exponents and terms

In [19]:
from generate_results import exponents_and_terms

exponents_and_terms()

"Dataset cubesat-primary, K = 1:\n\n| metric | exponent w |\n|---|---|\n| settling time | 0.1363 |\n| pointing error | 0.1644 |\n| total mass | 0.3577 |\n| control effort | below 1e-12 |\n| power margin | below 1e-12 |\n| energy balance | 0.1839 |\n| peak slew power | below 1e-12 |\n| depth of discharge | 0.1577 |\n\nDataset cubesat-primary, K = 3:\n\n| metric | term 1 | term 2 | term 3 |\n|---|---|---|---|\n| settling time | 0.3669 | 0.1848 | 0.05314 |\n| pointing error | 0.04362 | 0.08654 | 0.2265 |\n| total mass | below 1e-12 | 0.2405 | 0.05931 |\n| control effort | 0.3967 | below 1e-12 | below 1e-12 |\n| power margin | below 1e-12 | below 1e-12 | 0.02492 |\n| energy balance | 0.1928 | 0.4156 | below 1e-12 |\n| peak slew power | below 1e-12 | below 1e-12 | 0.6123 |\n| depth of discharge | below 1e-12 | 0.07253 | 0.0238 |\n\nDataset cubesat-primary, K = 3, per term:\n\n| term | rows | contending set | n_lemma | largest exponents | near-frontier | reach beta_k | share | gap G_k |\n|---|---|---|---|---|---|---|---|---|\n| 1 | 181 | control effort, settling time, energy balance, pointing error | 4 | control effort 0.397, settling time 0.367, energy balance 0.193 | 3 | 0.5664 | 0.03448 | below 1e-12 |\n| 2 | 162 | energy balance, total mass, settling time, pointing error, depth of discharge | 5 | energy balance 0.416, total mass 0.240, settling time 0.185 | 21 | 0.4952 | 0.2414 | below 1e-12 |\n| 3 | 363 | peak slew power, pointing error, settling time, total mass, energy balance | 5 | peak slew power 0.612, pointing error 0.227, total mass 0.059 | 73 | 0.545 | 0.7241 | below 1e-12 |\n\nTerm 1: too few near-frontier designs for an association reading: 3 of ten; its row above is the fit's, and its association cells are empty.\n\nDataset cubesat-stock, K = 1:\n\n| metric | exponent w |\n|---|---|\n| settling time | 0.3193 |\n| pointing error | 0.03927 |\n| total mass | below 1e-12 |\n| control effort | 0.4324 |\n| power margin | 0.1778 |\n| energy balance | 0.01936 |\n| peak slew power | below 1e-12 |\n| depth of discharge | 0.01195 |\n\nDataset cubesat-stock, K = 3:\n\n| metric | term 1 | term 2 | term 3 |\n|---|---|---|---|\n| settling time | 0.3681 | 0.07387 | 0.2773 |\n| pointing error | 0.04906 | 0.1896 | 0.006107 |\n| total mass | below 1e-12 | 0.1626 | below 1e-12 |\n| control effort | 0.5071 | below 1e-12 | 0.3529 |\n| power margin | below 1e-12 | 0.08397 | below 1e-12 |\n| energy balance | 0.07352 | 0 | 0.3477 |\n| peak slew power | below 1e-12 | 0.4237 | below 1e-12 |\n| depth of discharge | 0.002173 | 0.06638 | 0.01606 |\n\nDataset cubesat-stock, K = 3, per term:\n\n| term | rows | contending set | n_lemma | largest exponents | near-frontier | reach beta_k | share | gap G_k |\n|---|---|---|---|---|---|---|---|---|\n| 1 | 253 | control effort, settling time, energy balance, pointing error, depth of discharge | 5 | control effort 0.507, settling time 0.368, energy balance 0.074 | 20 | 0.3015 | 0.1154 | below 1e-12 |\n| 2 | 185 | peak slew power, pointing error, total mass, power margin, depth of discharge, settling time | 6 | peak slew power 0.424, pointing error 0.190, total mass 0.163 | 51 | 0.5928 | 0.4615 | below 1e-12 |\n| 3 | 342 | energy balance, control effort, settling time, depth of discharge | 4 | control effort 0.353, energy balance 0.348, settling time 0.277 | 50 | 0.2974 | 0.4231 | below 1e-12 |\n\nNo pair of traded metrics reaches |Spearman| 0.99 over the feasible designs, so no pair is read as one axis and every exponent is read on its own; the strongest pair: primary regime, power margin and energy balance, +0.982; stock regime, power margin and energy balance, +0.988. A pair at or above 0.99 would be named one axis and its two exponents read summed per term."

(sub-d-4)=
### D.4 The regimes side by side

In [20]:
from generate_results import regime_comparison

regime_comparison()

quantity,primary,stock
"tau, frac_above_, n_eff_","38.1, 0.092, 23.1","45.7, 0.104, 14.6"
"band 1/tau, near-frontier designs","0.0262, 56","0.0219, 77"
n_lemma,5,5
contending set,"total mass, energy balance, pointing error, depth of discharge, settling time","control effort, settling time, power margin, pointing error, depth of discharge"
"settling time, total mass",-0.50 (-0.50),-0.10 (+0.05)
"pointing error, control effort",-0.36 (-0.50),-0.63 (-0.69)
"settling time, pointing error",+0.03 (+0.15),+0.63 (+0.70)


(sub-d-5)=
### D.5 Sensitivity variants

In [21]:
from generate_results import sensitivity_variants

sensitivity_variants()

variant,setting,tau,frac_above_,n_eff_,n_lemma,contending set,largest exponent at,shared with baseline
baseline,"target 0.1, standard scaler, all",38.1,0.09207,23.1,5,"total mass, energy balance, pointing error, depth of discharge, settling time",total mass,5
target fraction halved,target 0.05,78.75,0.04816,8.89,5,"total mass, energy balance, pointing error, depth of discharge, settling time",total mass,5
target fraction doubled,target 0.2,16.83,0.1983,80.43,5,"total mass, energy balance, pointing error, depth of discharge, settling time",total mass,5
robust scaler,log-IQR scaling,78.75,0.09632,23.92,6,"peak slew power, total mass, pointing error, energy balance, depth of discharge, settling time",total mass,5
view: four metrics,4 attributes,10.69,0.0949,14.24,4,"pointing error, settling time, total mass, control effort",settling time,3
variant,setting,tau,frac_above_,n_eff_,n_lemma,contending set,largest exponent at,shared with baseline
baseline,"target 0.1, standard scaler, all",45.68,0.1038,14.6,5,"control effort, settling time, power margin, pointing error, depth of discharge",control effort,5
target fraction halved,target 0.05,78.75,0.04744,7.469,5,"control effort, settling time, power margin, pointing error, depth of discharge",control effort,5
target fraction doubled,target 0.2,22.1,0.2064,57.82,5,"control effort, settling time, power margin, pointing error, depth of discharge",control effort,5
robust scaler,log-IQR scaling,78.75,0.09231,10.51,5,"control effort, settling time, energy balance, depth of discharge, pointing error",control effort,4


(sub-d-6)=
### D.6 Refits by category

In [22]:
from generate_results import refits_by_category

refits_by_category()

wheel_class,designs,tau,frac_above_,n_eff_,n_lemma,contending set,largest exponent at
small,191,336.5,0.09424,5.467,6,"pointing error, total mass, peak slew power, depth of discharge, power margin, control effort",pointing error
medium,248,162.8,0.1048,8.784,6,"pointing error, total mass, peak slew power, depth of discharge, power margin, control effort",pointing error
large,267,113.2,0.1011,9.265,6,"pointing error, energy balance, peak slew power, total mass, depth of discharge, control effort",pointing error
wheel_class,designs,tau,frac_above_,n_eff_,n_lemma,contending set,largest exponent at
small,239,45.68,0.1004,6.965,6,"total mass, power margin, control effort, depth of discharge, pointing error, settling time",total mass
medium,279,65.68,0.09677,4.941,6,"power margin, control effort, total mass, settling time, pointing error, depth of discharge",control effort
large,262,78.75,0.09924,12.84,6,"control effort, settling time, power margin, total mass, pointing error, depth of discharge",control effort


(sub-d-7)=
### D.7 Candidate designs

In [23]:
from generate_results import candidate_designs

candidate_designs()

candidate,wheel_class,st_grade,log V,novelty V-1,rank among training,tau*log V
fast_effort_hungry,large,medium,-0.1973,-0.179,0.5326,-7.516
precise_heavy_sensor,large,fine,-0.153,-0.1419,0.619,-5.828
light_slow,small,coarse,-0.1709,-0.1571,0.5822,-6.511


(sub-d-8)=
### D.8 The solver's termination records

**What each status asserts.** `certified`: the assignment is fixed, residuals and gaps are within tolerance, no cell is empty and the margin is positive; levels 1 to 3 hold. `stationary_without_margin`: the same with a zero margin or an empty cell; levels 1 and 2 only. `uncertified`, `guard_stop` and `iteration_cap`: level 1 only, reported apart from certified. `not_applicable`: a penalty, the tangent step or a single term; the gaps are diagnostics. The record also holds the largest term gap, the Corollary bound, the margin, the stable radius and the shares.

In [24]:
from generate_results import termination_records

termination_records()

'Dataset cubesat-primary:\n\n| K | max gap | status | gamma | radius | bound |\n|---|---|---|---|---|---|\n| 1 | 8.696e-06 | not_applicable | nan | nan | 8.696e-06 |\n| 2 | below 1e-12 | certified | 0.000697 | 6.95e-05 | below 1e-12 |\n| 3 | below 1e-12 | certified | 0.00121 | 0.0001209 | below 1e-12 |\n\nDataset cubesat-stock:\n\n| K | max gap | status | gamma | radius | bound |\n|---|---|---|---|---|---|\n| 1 | 1.24e-05 | not_applicable | nan | nan | 1.24e-05 |\n| 2 | below 1e-12 | certified | 0.001413 | 0.0001327 | below 1e-12 |\n| 3 | below 1e-12 | certified | 0.00035 | 3.29e-05 | below 1e-12 |\n\nDataset planted-3-term, temperature 20:\n\n| K | tau | band 1/tau | objective_ | frac_above_ | term rows | term shares | max gap | status | gamma | radius | bound |\n|---|---|---|---|---|---|---|---|---|---|---|---|\n| 3 | 20 | 0.05 | 0.5863 | 0.3678 | [320, 340, 240] | [0.397, 0.328, 0.276] | below 1e-12 | certified | 0.0006768 | 2.864e-05 | below 1e-12 |\n\n| Planted terms | Largest exponent deviation after matching terms |\n|---|---|\n| 3 | 0.007605 |'

**How small is a small margin.** The margin gamma is in score units, the scale of the band 1/tau; below the band, a change as small as the band can move a design to another cell. Compare it with the median over the designs of the gap between best and second-best term score: far below it, a few designs sit on a seam and most do not. The stable radius is gamma divided by twice the largest norm of a row of the log-metrics with a final entry of minus one, so it is smaller than the margin and is sufficient, not tight.

In [25]:
from generate_results import margin_reading

margin_reading()

K,Margin (gamma),Stable radius,Band 1/tau,Median best-minus-second-best term score
2,0.000697,6.95e-05,0.0106,0.727
3,0.00121,0.0001209,0.0088,0.237
K,Margin (gamma),Stable radius,Band 1/tau,Median best-minus-second-best term score
2,0.001413,0.0001327,0.0139,0.233
3,0.00035,3.29e-05,0.0088,0.175


(sub-d-9)=
### D.9 Comparison with nondominated sorting

In [26]:
from generate_results import nondominated

nondominated()

view,columns,nondominated,of,share
all attributes,8,659,706,0.9334
four metrics,4,126,706,0.1785
view,columns,nondominated,of,share
all attributes,8,724,780,0.9282
four metrics,4,152,780,0.1949


(sub-d-10)=
### D.10 Propose, predict, check

In [27]:
from generate_results import propose_predict_check

propose_predict_check()

proposal,predicted log V,measured log V,difference,|diff| / band,log V within band,||dZ||,||dZ|| / band,Z within band
proposal_1,-0.09517,-0.04681,0.0484,0.517,True,0.09864,1.05,False
proposal_2,0.02481,0.0567,0.03189,0.3409,True,0.0473,0.506,True
proposal_3,-0.09517,-0.0468,0.0484,0.517,True,0.0986,1.05,False


(sub-d-11)=
### D.11 Stability of the conclusions under rounding-level jitter

In [28]:
from generate_results import stability

stability()

"The control effort and the saturated fraction are sums over the accepted steps of an adaptive integration and the settling time is detected on its dense solution, so a relative change of 1e-12 in a design variable moves them (gap G-17). Datasets cubesat-primary and cubesat-stock and the six check designs were regenerated 5 times with `Kp`, `Kd`, the panel span and the battery capacity of every design multiplied by 1 + e, e uniform in [-1e-12, 1e-12], and every quantity this paper reports was recomputed (685 quantities, of which 243 numbers move at all). The regenerations ran on the authors' machine and the committed data in the environment of record, so a range also holds the two platforms' difference. A quantity that moves is printed in this paper to the digits that held in all 6 datasets (the committed one and the 5 regenerated), as a range where not even two digits did, and as a bound where it is a numerical residue.\n\n| Conclusion | Quantities compared | Across the committed data and 5 regenerations |\n|---|---|---|\n| the contending sets (K = 1 and each term at K = 3) | 20 | identical |\n| each term's leading metrics | 8 | identical |\n| the signs of the reported tensions | 12 | identical |\n| the fit temperatures, to the printed 0.1 | 6 | identical |\n| the certificate statuses | 6 | identical |\n| the pass or miss of each check design | 6 | identical |\n| the count of nondominated designs on all attributes | 2 | identical |\n\nThe printed quantities that move, with the committed value, the minimum and maximum over the 6 datasets and the text this paper prints:\n\n| Quantity | Committed | Minimum | Maximum | Printed in this paper as |\n|---|---|---|---|---|\n| largest term gap, K = 2, primary | below 1e-12 | below 1e-12 | below 1e-12 | below 1e-12 |\n| margin, K = 2, primary | 0.0006969 | 0.0006968 | 0.000697 | 0.000697 |\n| radius, K = 2, primary | 6.948e-05 | 6.948e-05 | 6.95e-05 | 6.95e-05 |\n| largest term gap, K = 3, primary | below 1e-12 | below 1e-12 | below 1e-12 | below 1e-12 |\n| margin, K = 3, primary | 0.001213 | 0.001212 | 0.001213 | 0.00121 |\n| largest term gap, K = 2, stock | below 1e-12 | below 1e-12 | below 1e-12 | below 1e-12 |\n| largest term gap, K = 3, stock | below 1e-12 | below 1e-12 | below 1e-12 | below 1e-12 |\n| margin, K = 3, stock | 0.0003498 | 0.0003498 | 0.0003499 | 0.00035 |\n| radius, K = 3, stock | 3.286e-05 | 3.286e-05 | 3.287e-05 | 3.29e-05 |"

(sub-d-12)=
### D.12 The conditioning of the exponents

The diagnostic `conditioning_` of the estimator is the smallest eigenvalue of the restricted Hessian at each term's fit; a small value flags a nearly flat direction. It is read only and takes no part in a status.

*Why it matters.* The exponents of a term minimize a convex objective ({prf:ref}`thm-convex`), and `conditioning_` is that objective's curvature along its flattest direction: where it is small, moving the exponents along that direction barely changes the objective. The case has a large and a small value: the one-term fit has 7.1 in the primary regime and 0.017 in the stock regime (the K = 1 rows below).

*How it is computed.* `HyperbolicTradeSpace._conditioning` builds the Hessian of the objective at the fitted term, keeps the participating exponents and the simplex tangent space, and returns its smallest eigenvalue; `conditioning_diagnostic()` of `paper/generate_results.py` prints it per regime, K and term.

*Inspecting your own problem.* Fit the conditioned metrics at a finite temperature and read the attribute: `HyperbolicTradeSpace(temperature=50.0).fit(M).conditioning_` for a strictly positive array `M`. A small value says to look at the metrics themselves: rescale, merge or drop them, refit, and compare the value.

In [29]:
from generate_results import conditioning_diagnostic

conditioning_diagnostic()

K,Term,conditioning_ (smallest eigenvalue)
1,1,7.1
2,1,0.29
2,2,3.4
3,1,9.3
3,2,9
3,3,0.27
K,Term,conditioning_ (smallest eigenvalue)
1,1,0.017
2,1,5.3
2,2,9


(sub-d-13)=
### D.13 The refit on a term's own rows

{numref}`Section %s <sect-results>` says that refitting a term on its own rows is ill-posed. Here the two refits are compared with the true labels given, so the alternation plays no part. Script: `own_rows_refit` in `paper/generate_results.py`; the research spike that first measured it is `docs/superpowers/notes/spike-facet-refit/bench_oracle.py` (methods A and B).

In [30]:
from generate_results import own_rows_refit

own_rows_refit()

Temperature,Planted terms,Own-rows refit recovers,"Largest error, own rows",Support refit recovers,"Largest error, support"
infinity,60,0 of 60,0.97,60 of 60,0.001
50,60,0 of 60,0.96,60 of 60,0.037


(sub-d-14)=
### D.14 Does the fit recover a planted answer: twenty tables

{numref}`Section %s <sect-case-analysis>` asks whether the method recovers a known answer. The generator of {ref}`Appendix C <appendix-c>` plants three terms of known exponents; here it draws twenty tables of 400 rows and 4 metrics, each fitted at the exact maximum and at a working temperature, and each failure is listed with the frontier rows of its thinnest planted term. Script: `planted_recovery` in `paper/generate_results.py`; the research record is `docs/superpowers/notes/spike-facet-refit/`.

In [31]:
from generate_results import planted_recovery

planted_recovery()

"Dataset planted-3-term, the generator of Appendix C at 400 rows, 4 metrics and 3 terms, seeds 0 to 19, the solver's defaults (five restarts, `random_state = 0`). A table is recovered when the largest error of the exponents, after matching fitted to planted terms, is under 0.05 and at least 90% of its frontier rows lie on the right term.\n\n| Temperature | Planted tables | Recovered | Seeds not recovered |\n|---|---|---|---|\n| infinity | 20 | 16 of 20 | 1, 3, 8, 16 |\n| 50 | 20 | 16 of 20 | 1, 3, 8, 16 |\n\nThe tables not recovered, with the frontier rows of their thinnest planted term (a term needs at least 5 frontier rows, one more than there are metrics, to be determined by them), whether each was recovered at the exact maximum and at temperature 50, and at temperature 50 the solver's status, the largest exponent error and the share of frontier rows on the right term:\n\n| Seed | Frontier rows, thinnest term | Recovered at the exact maximum | Recovered at temperature 50 | Status at 50 | Largest error at 50 | Frontier rows on the right term at 50 |\n|---|---|---|---|---|---|---|\n| 1 | 5 | no | no | certified | 0.32 | 0.54 |\n| 3 | 6 | no | no | certified | 0.11 | 0.94 |\n| 8 | 11 | no | no | certified | 0.20 | 0.68 |\n| 16 | 5 | no | no | certified | 0.34 | 0.64 |\n\nAt the exact maximum the fit of each of these tables stops without a margin (`stationary_without_margin`) or at the iteration cap, depending on the platform, and its errors move with it, so neither is printed: a frontier row on two term planes ties at the exact maximum and the platform's argmax and linear-program vertex choice resolve the tie (gap G-10). Whichever status a fit stopped with, its largest term gap at termination was below `tol` (1e-10) in 40 of the 40 fits (level 2, numerical stationarity, L12), so a stop at the iteration cap is the labels not settling and not a term left unsolved; none of the fits stopped at the cap with a gap at or above `tol`. What does not depend on the platform is the count and the seeds: 16 of 20 tables were recovered, and the seeds above failed, at both temperatures on macOS arm64 and on Linux amd64. Every table recovered has at least 18 frontier rows on each planted term. At temperature 50 the status and the errors, printed to two digits, were the same on both platforms."

(sub-d-15)=
### D.15 The mean slack at a fixed temperature

The temperature is selected afresh at each K, so the mean slack of D.2 and of {numref}`section %s <sect-case-analysis>` compares fits at different temperatures. Here each K is refitted at the K = 1 temperature. `mean_slack_` is defined for every K and is comparable across K at one temperature; `objective_` is the soft maximum at one term and the mean slack at several, and is not.

In [32]:
from generate_results import mean_slack_fixed

mean_slack_fixed()

K,Temperature selected at this K,mean_slack_ there,"mean_slack_ refitted at the K = 1 temperature, 38.1"
1,38.1,0.243,0.243
2,94.4,0.182,0.132
3,113.2,0.146,0.088
K,Temperature selected at this K,mean_slack_ there,"mean_slack_ refitted at the K = 1 temperature, 45.7"
1,45.7,0.179,0.179
2,71.9,0.142,0.114
3,113.2,0.093,0.046


(sub-d-16)=
### D.16 Chaining with PCA: the principal directions of the near-frontier chart

The tension report runs a principal component analysis on the chart coordinates Z of the near-frontier designs in a term's cell. Its directions say along which combinations of metrics the near-frontier designs vary the most, with loadings on the metrics through the chart basis; the fit says which metrics bind, the analysis says where the sampled designs spread along the fitted surface. They answer different questions and the second is computed from the first's coordinates. The sign of a direction is arbitrary; each is shown with its largest loading positive.

In [33]:
from generate_results import pca_chain

pca_chain()

Direction,Share of the spread of the chart coordinates,Leading loadings on the metrics
1,0.444,"total mass +0.53, control effort +0.53, pointing error -0.47"
2,0.238,"pointing error +0.79, depth of discharge -0.38, energy balance -0.26"
3,0.159,"depth of discharge +0.69, power margin -0.49, energy balance -0.46"


(sub-d-17)=
### D.17 The requirements read after the fit

The fit of record is the fit on the feasible designs, with no threshold requirement imposed before it. This section reads the model's four threshold requirements after the fit: peak slew power 2.6 W, pointing error 0.008495 degree, control effort 0.05922 N·m·s and total mass 15.107 kg (the launch budget of 90,643 USD at the illustrative 6,000 USD/kg), part `budgets` of `cubesat.sysml`. The first table prices each requirement at the fitted K = 1 frontier (`case.requirements_table`): the share of the feasible designs that pass it, the near-frontier designs it excludes, the metric's exponent, the price in log V of meeting it for the median excluded near-frontier design (the exponent times the needed tightening in the conditioned metric, also in band widths) and the two largest elasticities along the surface, the percent change in another metric per percent change in this one, for metrics of exponent at least 0.05. A requirement on a metric at zero exponent has no price at K = 1: the fitted frontier does not trade it. The price is a feature of the fitted model on this table, not a causal exchange.

In [34]:
from generate_results import requirements_at_the_frontier

requirements_at_the_frontier()

requirement,limit,pass (feasible),near-frontier excluded,K = 1 exponent,price (log V),price (bands),largest elasticities
peak slew power,2.6 W,62.2 %,0 of 56,0.000,0.0000,0.00,none at K = 1 (exponent 0)
pointing error,0.008495 degree,95.0 %,1 of 56,0.164,0.0019,0.07,settling time -1.16; energy balance +0.86
control effort,0.05922 N.m.s,58.9 %,18 of 56,0.000,0.0000,0.00,none at K = 1 (exponent 0)
total mass,15.107 kg,94.9 %,0 of 56,0.358,0.0000,0.00,none excluded
all jointly,,40.8 %,19 of 56,,,,
requirement,limit,pass (feasible),near-frontier excluded,K = 1 exponent,price (log V),price (bands),largest elasticities
peak slew power,2.6 W,67.3 %,2 of 77,0.000,0.0000,0.00,none at K = 1 (exponent 0)
pointing error,0.008495 degree,63.7 %,42 of 77,0.039,0.0122,0.56,power margin +0.16; settling time -0.15
control effort,0.05922 N.m.s,90.0 %,1 of 77,0.432,0.1734,7.92,power margin +1.40; settling time -1.36
total mass,15.107 kg,95.6 %,1 of 77,0.000,0.0000,0.00,none at K = 1 (exponent 0)


The second table fits three design sets per dataset: every design, the feasible designs (the fit of record) and the feasible designs that also meet the four requirements, the constrained set (`case.design_sets_table`, `case.design_sets_placement`). Every set is conditioned by the feasible set's conditioning, so a difference between the fits is the rows', not a re-scaling's; the temperature is selected on each set's rows by the same rule. The zero set holds the metrics at zero exponent at every temperature of the sweep. The research record of both tables is `work-packages/WP23-rounds/spikes/`.

In [35]:
from generate_results import design_sets_comparison

design_sets_comparison()

"Datasets cubesat-primary, cubesat-stock. The fit on each design set:\n\n| dataset | set | designs | tau | contending set | leading exponents | zero set | three terms (share) |\n|---|---|---|---|---|---|---|---|\n| cubesat-primary | all designs | 1000 | 31.8 | total mass, energy balance, pointing error, depth of discharge, settling time | total mass 0.357, energy balance 0.181, pointing error 0.163 | power margin, peak slew power | settling time 0.406, control effort 0.387 (0.05) \\| peak slew power 0.703, pointing error 0.237 (0.73) \\| energy balance 0.395, total mass 0.247 (0.22) |\n| cubesat-primary | feasible | 706 | 38.1 | total mass, energy balance, pointing error, depth of discharge, settling time | total mass 0.358, energy balance 0.184, pointing error 0.164 | power margin, peak slew power | control effort 0.397, settling time 0.367 (0.03) \\| energy balance 0.416, total mass 0.240 (0.24) \\| peak slew power 0.612, pointing error 0.227 (0.72) |\n| cubesat-primary | requirements met | 288 | 65.7 | total mass, energy balance, pointing error, depth of discharge, settling time | total mass 0.357, energy balance 0.184, pointing error 0.177 | control effort, power margin, peak slew power | peak slew power 0.450, pointing error 0.244 (0.09) \\| peak slew power 0.413, pointing error 0.209 (0.18) \\| peak slew power 0.713, pointing error 0.236 (0.73) |\n| cubesat-stock | all designs | 1000 | 29.0 | control effort, settling time, power margin, pointing error, total mass | control effort 0.414, settling time 0.334, power margin 0.190 | peak slew power | peak slew power 0.576, pointing error 0.212 (0.69) \\| control effort 0.501, settling time 0.402 (0.06) \\| control effort 0.351, energy balance 0.317 (0.25) |\n| cubesat-stock | feasible | 780 | 45.7 | control effort, settling time, power margin, pointing error, depth of discharge | control effort 0.432, settling time 0.319, power margin 0.178 | peak slew power | control effort 0.507, settling time 0.368 (0.12) \\| peak slew power 0.424, pointing error 0.190 (0.46) \\| control effort 0.353, energy balance 0.348 (0.42) |\n| cubesat-stock | requirements met | 302 | 54.8 | control effort, settling time, energy balance, pointing error, depth of discharge | control effort 0.441, settling time 0.305, energy balance 0.212 | total mass, power margin, peak slew power | control effort 0.293, energy balance 0.272 (0.60) \\| control effort 0.461, settling time 0.300 (0.00) \\| peak slew power 0.356, total mass 0.216 (0.40) |\n\nThe baseline configuration and the check designs in each set's one-term fit, log V (in band widths, share of the set's designs below):\n\n| dataset | design | passes the flags | meets the requirements | fit on all designs | fit on feasible | fit on requirements met |\n|---|---|---|---|---|---|---|\n| cubesat-primary | baseline configuration | True | False | -0.499 (-15.9, 0.09) | -0.516 (-19.6, 0.10) | -0.566 (-37.1, 0.01) |\n| cubesat-primary | fast_effort_hungry | True | False | -0.166 (-5.3, 0.58) | -0.197 (-7.5, 0.53) | -0.257 (-16.9, 0.22) |\n| cubesat-primary | precise_heavy_sensor | True | False | -0.125 (-4.0, 0.68) | -0.153 (-5.8, 0.62) | -0.191 (-12.6, 0.36) |\n| cubesat-primary | light_slow | True | False | -0.147 (-4.7, 0.63) | -0.171 (-6.5, 0.58) | -0.227 (-14.9, 0.27) |\n| cubesat-primary | proposal_1 | False | False | -0.543 (-17.2, 0.08) | -0.570 (-21.7, 0.07) | -0.591 (-38.8, 0.01) |\n| cubesat-primary | proposal_2 | False | False | -0.630 (-20.0, 0.04) | -0.660 (-25.2, 0.04) | -0.688 (-45.2, 0.00) |\n| cubesat-primary | proposal_3 | False | False | -0.543 (-17.2, 0.08) | -0.570 (-21.7, 0.07) | -0.591 (-38.8, 0.01) |\n| cubesat-stock | baseline configuration | True | False | -0.075 (-2.2, 0.69) | -0.057 (-2.6, 0.75) | -0.030 (-1.6, 0.80) |\n| cubesat-stock | fast_effort_hungry | True | False | -0.125 (-3.6, 0.53) | -0.200 (-9.1, 0.32) | -0.196 (-10.7, 0.38) |\n| cubesat-stock | precise_heavy_sensor | True | False | +0.031 (+0.9, 0.95) | -0.064 (-2.9,

(sub-d-18)=
### D.18 The noise-ridge corollary against injected noise

{numref}`Section %s <sect-results>` states that noise on the metrics shifts the exponents by a ridge (L20). This section checks the first-order shift against noise injected into the cubesat data, in both regimes. The predicted shift grows as `conditioning_` falls and measured shifts agree with it within the spread of the draws at half a percent in both regimes and at 2 percent in the primary one; where the predicted shift exceeds the smallest active exponent the formula is outside its range, the contending set changes (the Jaccard column) and the measured shift saturates. The conditioning is refitted on each noisy table, which L18 does not cover, and eight draws are few.

In [36]:
from generate_results import injected_noise

injected_noise()

regime,conditioning_,delta (%),predicted largest shift,measured largest shift (mean),spread of the draws,contending-set Jaccard
cubesat-primary,7.092,0.5,0.003,0.005,0.012,0.94
cubesat-primary,7.092,2.0,0.047,0.071,0.043,0.82
cubesat-primary,7.092,5.0,0.292,0.275,0.098,0.71
cubesat-primary,7.092,10.0,1.168,0.304,0.067,0.68
cubesat-stock,0.017,0.5,0.022,0.032,0.047,0.98
cubesat-stock,0.017,2.0,0.348,0.102,0.068,0.98
cubesat-stock,0.017,5.0,2.174,0.136,0.035,0.93
cubesat-stock,0.017,10.0,8.694,0.147,0.043,0.89


(appendix-e)=
## Appendix E. Assumptions and abstraction gaps

**Job.** To say under what assumptions the formal statements carry over to data, and how sensitive the results are to each.

- **Main text keeps:** the paragraph "Map and territory" in {numref}`section %s <sect-notation>`, each simplifying assumption named and defended where it is made, and the future work organized by them.
- **This appendix carries:** one table with a row per gap, G-1 to G-17: the formal object, the bridging assumption, what cannot be proved, the sensitivity evidence and the interpretive consequence, compiled from the ledger.
- **The repository keeps:** the ledger with its evidence (`docs/superpowers/notes/abstraction-gaps.md`).

In [37]:
from generate_gaps import gaps_table

gaps_table()

Gap,Formal object,Bridging assumption,What cannot be proved,Sensitivity evidence,Interpretive consequence
G-1,"G-1. The three objects. (a) The fitted reference surface: $\Phi(u)=w\cdot u-\beta=0$ at finite $\tau$ with $\beta=J_\tau(Uw)$ (L3, L5); $V>1$ is possible. (b) The $\tau=\infty$ supporting envelope: $\max_i w\cdot u_i=\beta$, the LP optimum (I1); no observed row above it. (c) The Pareto boundary of the attainable set, a set the data sample. L11: a $V$-maximizer is weakly efficient; the converse is not claimed.",The sampled designs reach the boundary of the attainable set (the data are near-frontier somewhere); $\tau$ is chosen so a stated fraction of rows lies above $V=1$; only the log-convex part of a Pareto front is reached by some $w$ (L11).,That (a) or (b) lies within a bound of (c); that the sample reaches (c); that (a) at the selected $\tau$ is the same set as (b).,"Fraction above $V=1$ is a property of $(\tau,K)$: at $\tau=38.10$ it is 0.092, 0.190, 0.261 for $K=1,2,3$ (primary), and 0.104, 0.176, 0.268 at $\tau=45.68$ (baseline); re-selecting $\tau$ per $K$ returns it to 0.092 to 0.104 (examples/adcs/k_terms.py on data/v2/, the fixed-$\tau$ diagnostic). The Pareto boundary of the cubesat attainable set is not observed. tests/test_derivation.py::test_L11_in_chain_and_states_non_converse.","$V=1$ reads as the reference level of a fitted object; $V>1$ is beyond the fitted reference surface, a candidate to test, not a point proved attainable or unattainable; Pareto statements are made only for maximizers, through L11."
G-2,"G-2. Existence, uniqueness, stability of $w$. Existence: continuous $J_\tau$ on the compact simplex. Uniqueness: strict convexity on the simplex tangent space, the restricted Hessian $\tau(U^\top\mathrm{diag}(p)U-(U^\top p)(U^\top p)^\top)$ positive definite on the participating coordinates (L3, L4, R1). Stability: the smallest eigenvalue of that matrix.",An exponent above the participation threshold at the fit temperature participates; an exponent at or below $10^{-12}$ is mathematically zero; the restricted Hessian at the fit is positive definite.,That a given data set has a positive definite restricted Hessian; that numerically zero is mathematically zero; that $10^{-3}$ is the right participation level for a given application.,"Cubesat conditioning_ on the eight-metric data (M3): $K=1$ 7.09 (primary), 0.0170 (baseline); $K=3$ primary 9.03, 9.26, 0.273 (the terms led by energy balance, control effort and peak slew power); baseline 9.32, 0.228, 10.2. Halving or doubling $\tau$ at $K=1$ moves the largest exponent by 0.008 and 0.001 (primary, five participants throughout), 0.004 and 0.001 (baseline, six participants throughout) (M5). The primary $K=3$ term led by peak slew power has 6, 6, 4 exponents above $10^{-3}$, $10^{-2}$, $0.03$ (M3). tests/test_estimator.py::test_conditioning_drops_for_nearly_collinear_metrics (conditioning below $10^{-2}$ of the base at $10^{-3}$ log-noise collinearity); docs/superpowers/notes/adcs-tensions-probe.md (exponents below about 0.03 move with a few vertex rows).","The contending set is read as the metrics above the threshold, and an exponent near the threshold is read as undetermined; a small conditioning_ marks a flat direction in which two fits with equal objective differ in $w$."
G-3,"G-3. L6 population identity against a finite sample. L6: under A5 ($\varepsilon_j$ i.i.d. $N(0,\sigma^2)$, $\sigma>0$) and I5, $\tfrac1\tau\log E\,e^{\tau w\cdot\varepsilon}=\tau\sigma^2\lVert w\rVert^2/2$, an identity for the population expectation; with a noise covariance $\Sigma$ in place of $\sigma^2I$ the cumulant is $\tau w^\top\Sigma w/2$ (L18), and the effect of that noise on the exponents is L20's first-order ridge shift.","The conditioned log-noise is i.i.d. Gaussian and independent of the design (A5); for any finite-sample or consistency statement, independence, integrability and identifiability of $w$ hold.","Unbiasedness or consistency of the fitted offset; that t

(appendix-f)=
## Appendix F. Computational reproducibility

**Job.** To let a reader regenerate every figure, table and reported number in the paper, in the same environment the authors used, without first learning the repository.

- **Main text keeps:** {numref}`section %s <sect-reproducibility>` in a few sentences: everything is regenerable, the environment and the commands are in this appendix, use of the package on one's own table is on the docsite.
- **This appendix carries:** the pinned toolchain generated from the lock, the environment of record, the commands and what a pass looks like, and one row per figure and table with its call, its input and its output.
- **The repository keeps:** the code, the tests, the lock file, the Dockerfile and the ledger.

(sub-f-1)=
### F.1 The pinned toolchain

In [37]:
from generate_repro import toolchain

toolchain()

Item,Value,Pinned by
Python,3.12,".python-version; requires-python >=3.12,<3.13"
Package manager,uv 0.9.18,uv.lock (format revision 3)
Locked packages,178,uv.lock
numpy,2.5.4,uv.lock; runtime dependency
scipy,1.18.1,uv.lock; runtime dependency
sympy,1.14.0,uv.lock; runtime dependency
scikit-learn,1.9.1,uv.lock; runtime dependency
pandas,3.1.1,uv.lock; runtime dependency
matplotlib,3.11.2,uv.lock; runtime dependency
hyperbolic-trade-space,0.0.1,pyproject.toml; this repository


(sub-f-2)=
### F.2 The environment of record

% container-paragraph:begin

Inside the pinned linux/arm64 image the results reproduce byte for byte. Between that image and the Mac, and between it and a comparable Ubuntu image, values differ at the last digits because the platforms' mathematical libraries differ, and the conclusions are unchanged. The exact tests therefore run only in the environment of record, and everything else compares within stated tolerances. The image id changes with every commit because the history layer carries the commit. Git history enters the image through a git bundle that `paper/container/build.sh` passes in. The image pins Node.js for arm64, with its checksum and signature verified.

% container-paragraph:end

(sub-f-3)=
### F.3 Commands

The tests assume a git checkout of the repository: a few read its history, and in a copy made without it they skip with a reason. The last row of the table regenerates one row of F.4, the cells it needs included.

In [38]:
from generate_repro import commands

commands()

Command,What it does,A pass looks like
uv sync --locked,install the locked environment,exits 0
uv lock --check,check that the lock matches pyproject.toml,exits 0
uv run pytest,"run the test suite, doctests included",all tests pass
uv run python -m derivation.run,run the chain L1 to L20,every lemma reports PASS
uv run python -m derivation.proofmap,check the proof map against the scripts,exits 0
uv run python paper/execute_notebooks.py,execute both notebooks from a clean kernel and write the outputs,"exits 0; the outputs are unchanged when the data and the seeds are: git diff --stat -- paper/paper.ipynb paper/supporting_material.ipynb then shows only the commit and platform lines of the toolchain table, and PAPER_EXECUTE=1 uv run pytest tests/test_paper.py -k clean_kernel compares every other text output of a fresh run with the committed one"
uv run python paper/regenerate_row.py ROW,"regenerate one row of the table in F.4 (ROW is its first column): runs the cells that row needs and prints the table or saves the figure, writing nothing in the repository","prints the table, equal to that cell's committed output, or names the figure file"
cd paper && uv run myst build --html,"build the paper as a static site, written to paper/_build/html","exits 0, with no error reported; it prints a line that a server started on a local port, because the builder renders the pages through a temporary server that stops when the build ends"


(sub-f-4)=
### F.4 One row per figure and table

In [40]:
from generate_repro import reproduction_table

reproduction_table()

Item,Call,Reads,Writes
tab-families,family_table(),paper/prior_art.yaml,cell tab-families of paper.ipynb
fig-tau-sweep,"case.tau_sweep_figure(study, fit)",examples/adcs/data/v2/ (the regimes' CSV files named by the case module),cell fig-tau-sweep of paper.ipynb
fig-tensions-k3,"case.tension_figure(study, fit3, band=band)",examples/adcs/data/v2/ (the regimes' CSV files named by the case module),cell fig-tensions-k3 of paper.ipynb
tab-A-register,register_table(),glossary.yaml,cell tab-A-register of supporting_material.ipynb
tab-B-how-to-read,how_to_read(),derivation/proof_map_record.json,cell tab-B-how-to-read of supporting_material.ipynb
tab-B-proof-map,proof_map(),derivation/proof_map_record.json,cell tab-B-proof-map of supporting_material.ipynb
tab-B-proofs-fit,proofs_fit(),derivation/proof_map_record.json,cell tab-B-proofs-fit of supporting_material.ipynb
tab-B-proofs-solver,proofs_solver(),derivation/proof_map_record.json,cell tab-B-proofs-solver of supporting_material.ipynb
tab-B-checker,checker(),derivation/proof_map_record.json,cell tab-B-checker of supporting_material.ipynb
tab-C-configuration_space,configuration_space(),examples/adcs/data/v2/ (the regimes' CSV files named by the case module),cell tab-C-configuration_space of supporting_material.ipynb


(appendix-g)=
## Appendix G. Register of prior art

**Job.** To show that the homework was done, and answer in advance the objection that this work is just some existing method.

- **Main text keeps:** {numref}`section %s <sect-prior-art>` in about 600 words: the family table, the Pareto and DEA paragraphs and the lineage.
- **This appendix carries:** the larger register, grouped by family, one entry per work or family with four fields (what it does; what this work shares with it or takes from it; what this work does differently, extends or adds; the citations), compiled from `paper/prior_art.yaml`; and the reading pass, a table with a row for each sentence of the paper that says what a cited work says: what was read, the verdict and what the paper now does with the sentence, compiled from `paper/reading_pass.yaml`.
- **The repository keeps:** the long review with its dropped candidates (`docs/superpowers/notes/paper-sources/literature.md`) and the bibliography of all adopted sources.

This register is not exhaustive. Its entries are the works and families most immediately relevant to the method, and each says what this work does differently, extends or adds. The bibliographic records of the cited works were verified; what each work says was read only in part, and part G.2 below reports what was read.

(sub-g-1)=
### G.1 The register

In [40]:
from generate_prior_art import register

register()

'\n#### Optimization and efficiency analysis\n\n**Pareto and multiobjective optimization**\n\n- *What it does.* Returns a nondominated set of designs, and finds Pareto-optimal solutions by scalarization, for example the weighting method; evolutionary algorithms generate nondominated sets.\n- *What this work shares with it or takes from it.* A maximizer of a weighted geometric mean with positive exponents is Pareto-efficient (lemma L11), and the weights act as a trade-off.\n- *What this work does differently, extends or adds.* Returns no coordinate system and no exponents: a nondominated set says which designs survive, the fit says what trades against what, with one exponent vector per term, a band of near-boundary designs and an effective sample size. Dominance is a hard comparison, the fit a soft maximum. Identification from a table, not generation of designs.\n- *Why this is not just the neighbour.* This is not just a Pareto front: a front is the set of designs that no other design dominates, while the fit returns a reference surface with coordinates, and for each term the exponents that give the elasticities between metrics.\n- *Cites.* <a href="../#cite-miettinen1999nonlinear" target="_self" style="color:inherit;text-decoration:underline">Miettinen 1999</a>; <a href="../#cite-ehrgott2005multicriteria" target="_self" style="color:inherit;text-decoration:underline">Ehrgott 2005</a>; <a href="../#cite-zitzler1999multiobjective" target="_self" style="color:inherit;text-decoration:underline">Zitzler and Thiele 1999</a>; <a href="../#cite-deb2002fast" target="_self" style="color:inherit;text-decoration:underline">Deb et al. 2002</a>\n\n**Data envelopment analysis (DEA)**\n\n- *What it does.* Evaluates the relative efficiency of each decision-making unit by mathematical programming on the observed data, with weights obtained from the data, with variants for scale and returns to scale, a super-efficiency variant that ranks efficient units against a reference technology spanned by all other units, and Malmquist productivity indices across data sets.\n- *What this work shares with it or takes from it.* A one-sided envelope of the data, a score per unit, scores beyond the boundary (V above 1) and a boundary that moves between data sets.\n- *What this work does differently, extends or adds.* One exponent vector per term, a global named structure, in place of weights obtained unit by unit; a surface that is piecewise log-linear, with constant elasticities within a term; larger-is-better metrics with homogeneity by the simplex in place of inputs, outputs and a returns-to-scale choice; a soft maximum with a temperature, a band and an effective sample size in place of a deterministic boundary set by extreme units; one convex program and K refits; coordinates and a pull-back to attribute units in place of a score and a peer set. In short, global exponents per term, not per-unit weights; a band, not extreme units.\n- *Why this is not just the neighbour.* This is not just DEA: DEA scores each unit against the envelope under weights obtained from the data, while the fit learns one set of exponents per term from the whole table and reports where a design sits relative to its term and which metrics trade there.\n- *Cites.* <a href="../#cite-charnes1978measuring" target="_self" style="color:inherit;text-decoration:underline">Charnes et al. 1978</a>; <a href="../#cite-banker1984some" target="_self" style="color:inherit;text-decoration:underline">Banker et al. 1984</a>; <a href="../#cite-andersen1993procedure" target="_self" style="color:inherit;text-decoration:underline">Andersen and Petersen 1993</a>; <a href="../#cite-fare1994productivity" target="_self" style="color:inherit;text-decoration:underline">Färe et al. 1994</a>; <a href="../#cite-allen1997weights" target="_self" style="color:inherit;text-decoration:underline">Allen et al. 1997</a>; <a href="../#cite-dyson2001pitfalls" target="_self" style="color:inherit;text-decoration:underline">Dys

(sub-g-2)=
### G.2 What the cited works say: the reading pass

A fresh reader read what is accessible of each cited work (abstracts, open full texts, publisher descriptions; not the paywalled books and articles) and checked each sentence of {numref}`section %s <sect-problem>` and {numref}`section %s <sect-prior-art>`, the sketch of the Pareto theorem (L11) and the process paragraph that says what a cited work says. A verdict is *supported* (the text read says what the sentence attributes to it), *partly supported* (some of it, or only the topic), *not found* (what was reachable does not say it) or *not accessible*. Most verdicts rest on abstracts. A sentence was kept where the verdict is supported; reworded to what was read where it is partly supported or not found; and, where the source was not accessible, reworded to what its record supports and marked *to be confirmed by the author*. The sentence on the author's INCOSE paper is listed for him and unchanged.

In [41]:
from generate_reading_pass import reading_pass_table

reading_pass_table()

#,Sentence,Key,What was read,Verdict,In the paper
1,"A decision analysis has two products: an answer, and a justification that an accountable executive can follow and defend",Howard 1988,"Crossref API https://api.crossref.org/works/10.1287/mnsc.34.6.679, abstract (full)",partly supported,"reworded to what was read. The two products are the paper's own statement, uncited; the work is cited for a clear conversation with the decision-maker."
2,(same sentence),Keeney and Raiffa 1993,"https://www.cambridge.org/core/books/decisions-with-multiple-objectives/DEF338459C327778C3F8C4C4A682032F, publisher description; Crossref API https://api.crossref.org/works/10.1017/CBO9781139174084, abstract",partly supported,reworded to what was read. Cited now for responsible choice among conflicting objectives.
3,"In the practice we know a trade study is a document: a grid, a table, a ranking by weighted criteria",Baker and Whalen 2012,"https://www.nasa.gov/wp-content/uploads/2016/10/survey_of_trade_study_methods_-_baker.pdf, whole (a one-page poster) read",partly supported,reworded to what was read. Cited now only for 'surveys compare them' (the weighted-criteria methods); that a trade study is a document is stated as the practice the author knows.
4,(same sentence),Parnell 2016,"https://www.wiley.com/en-us/Trade+off+Analytics%3A+Creating+and+Exploring+the+System+Tradespace-p-9781119237532, publisher description",not found,reworded to what was read. Removed from the sentence; the work stays in the bibliography and in Table 1 as a decision-management text.
5,"The weights are human judgments, and human judgment in trade studies is open to known biases",Smith et al. 2007,"Crossref API https://api.crossref.org/works/10.1002/sys.20072, abstract",supported,kept as read.
6,(same sentence),Chilli et al. 2026,"https://sebokwiki.org/wiki/Decision_Management, article body (fetched)",partly supported,"reworded to what was read. Cited together with Smith for bias in trade studies, as the SEBoK article says it can distort decisions; no longer cited for the weights."
7,Systems engineering is moving from documents to models,Madni and Sievers 2018,"Crossref API https://api.crossref.org/works/10.1002/sys.21438, abstract",partly supported,reworded to what was read. Cited only for the move to model-based practice centred on the system model; the move from documents is cited to Henderson alone.
8,(same sentence),Henderson and Salado 2021,"Crossref API https://api.crossref.org/works/10.1002/sys.21566, abstract",supported,kept as read.
9,"PCA has that standing [recognized, reproducible, rigorous, trusted]",Jolliffe and Cadima 2016,"Crossref API https://api.crossref.org/works/10.1098/rsta.2015.0202, abstract",partly supported,"reworded to what was read. Cited now for what PCA does (reduces dimensionality, increases interpretability, minimizes information loss); the standing is the paper's own wording."
10,"Table row Pareto: 'Returns a nondominated set of designs, and traces its supported part with scalarizations such as weighted sums' (and Pareto paragraph: 'Pareto methods return a nondominated set and trace its supported part by scalarization')",Miettinen 1999,https://users.jyu.fi/~miettine/book/ (author's overview and contents); Springer page redirects to a cookie gate,partly supported,"reworded to what was read. The words 'supported part' are gone; the sentence says scalarization, for example the weighting method, finds Pareto-optimal solutions."
